# Pokemon VAE — Training Notebook

End-to-end training of a Variational Autoencoder on 96×96 RGBA Pokemon sprites.

**Before running:** populate the data splits from the project root:
```bash
python data/process_data.py
```

In [ ]:
import csv
import os
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

## Configuration

In [ ]:
@dataclass
class VAEConfig:
    # Paths — relative to the project root (where this notebook lives)
    data_dir: str = "data"
    checkpoint_dir: str = "checkpoints"
    log_dir: str = "logs"

    # Image
    image_size: int = 96
    channels: int = 4           # RGBA

    # Model
    latent_dim: int = 128
    hidden_dim: int = 512
    beta: float = 1.0           # KL weight; 1.0 = standard VAE
    encoder_dropout: float = 0.3

    # LR scheduler (ReduceLROnPlateau)
    lr_patience: int = 15
    lr_factor: float = 0.5
    lr_min: float = 1e-6

    # Training
    batch_size: int = 64
    epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    num_workers: int = 4
    pin_memory: bool = True

    # Checkpointing
    save_every: int = 10
    resume_from: str = ""       # set to a .pt path to resume training

    # Logging
    log_interval: int = 50      # print loss every N batches

cfg = VAEConfig()

## Dataset

In [ ]:
def get_transforms(image_size: int, augment: bool = False) -> transforms.Compose:
    ops = []
    if augment:
        ops.append(transforms.RandomHorizontalFlip())
    ops += [
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),  # scales uint8 → float [0, 1]
    ]
    return transforms.Compose(ops)


class PokemonDataset(Dataset):
    def __init__(self, data_dir: str, image_size: int = 96, augment: bool = False):
        self.paths = sorted(Path(data_dir).glob("*.png"))
        if not self.paths:
            raise FileNotFoundError(f"No PNG files found in {data_dir}")
        self.transform = get_transforms(image_size, augment)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.paths[idx]).convert("RGBA")
        return self.transform(img)

In [ ]:
def build_loader(cfg: VAEConfig, split: str, augment: bool) -> DataLoader:
    path = os.path.join(cfg.data_dir, split)
    dataset = PokemonDataset(path, image_size=cfg.image_size, augment=augment)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=(split == "training"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=(split == "training"),
    )

train_loader = build_loader(cfg, "training",   augment=True)
val_loader   = build_loader(cfg, "validation", augment=False)
test_loader  = build_loader(cfg, "test",       augment=False)

print(f"Train : {len(train_loader.dataset):>5} images")
print(f"Val   : {len(val_loader.dataset):>5} images")
print(f"Test  : {len(test_loader.dataset):>5} images")

sample = next(iter(train_loader))
print(f"\nBatch shape : {sample.shape}")
print(f"dtype / range: {sample.dtype}  [{sample.min():.2f}, {sample.max():.2f}]")

## VAE Architecture

In [ ]:
class Encoder(nn.Module):
    def __init__(self, channels: int, hidden_dim: int, latent_dim: int, dropout: float = 0.3):
        super().__init__()
        # 96 → 48 → 24 → 12 → 6
        self.conv = nn.Sequential(
            nn.Conv2d(channels, 32, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.flat_dim = 256 * 6 * 6
        self.fc        = nn.Linear(self.flat_dim, hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc_mu     = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        x = self.conv(x).flatten(1)
        x = self.dropout(torch.relu(self.fc(x)))
        return self.fc_mu(x), self.fc_log_var(x)


class Decoder(nn.Module):
    def __init__(self, channels: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.flat_dim = 256 * 6 * 6
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, self.flat_dim),
            nn.ReLU(inplace=True),
        )
        # 6 → 12 → 24 → 48 → 96
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, channels, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, 256, 6, 6))


class VAE(nn.Module):
    def __init__(self, channels: int = 4, hidden_dim: int = 512, latent_dim: int = 128,
                 dropout: float = 0.3):
        super().__init__()
        self.encoder = Encoder(channels, hidden_dim, latent_dim, dropout)
        self.decoder = Decoder(channels, hidden_dim, latent_dim)

    def reparameterize(self, mu, log_var):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * log_var)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        return self.decoder(self.reparameterize(mu, log_var)), mu, log_var

    def sample(self, n: int, device: torch.device) -> torch.Tensor:
        z = torch.randn(n, self.encoder.fc_mu.out_features, device=device)
        return self.decoder(z)


def vae_loss(recon, target, mu, log_var, beta=1.0):
    recon_loss = nn.functional.mse_loss(recon, target, reduction="sum") / target.size(0)
    kl_loss    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / target.size(0)
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

## Training

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, loss, path):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "loss": loss,
    }, path)
    print(f"  Checkpoint saved → {path}")


def load_checkpoint(path, model, optimizer, scheduler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    print(f"Resumed from epoch {ckpt['epoch']}  (loss={ckpt['loss']:.4f})")
    return ckpt["epoch"]

In [ ]:
def train_epoch(model, loader, optimizer, device, cfg, epoch):
    model.train()
    total = 0.0
    for i, batch in enumerate(loader):
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, log_var = model(batch)
        loss, recon_l, kl_l = vae_loss(recon, batch, mu, log_var, cfg.beta)
        loss.backward()
        optimizer.step()
        total += loss.item()
        if (i + 1) % cfg.log_interval == 0:
            print(f"  [epoch {epoch}  batch {i+1}/{len(loader)}]"
                  f"  loss={loss.item():.4f}  recon={recon_l.item():.4f}  kl={kl_l.item():.4f}")
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, cfg):
    model.eval()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        recon, mu, log_var = model(batch)
        loss, _, _ = vae_loss(recon, batch, mu, log_var, cfg.beta)
        total += loss.item()
    return total / len(loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.log_dir).mkdir(parents=True, exist_ok=True)

model = VAE(
    channels=cfg.channels,
    hidden_dim=cfg.hidden_dim,
    latent_dim=cfg.latent_dim,
    dropout=cfg.encoder_dropout,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=cfg.lr_factor,
    patience=cfg.lr_patience, min_lr=cfg.lr_min,
)

start_epoch = 0
if cfg.resume_from:
    start_epoch = load_checkpoint(cfg.resume_from, model, optimizer, scheduler, device)

print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
log_path = os.path.join(cfg.log_dir, "train_log.csv")
with open(log_path, "w") as f:
    f.write("epoch,train_loss,val_loss,lr\n")

val_loss = float("inf")
for epoch in range(start_epoch + 1, cfg.epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, device, cfg, epoch)
    val_loss   = eval_epoch(model, val_loader, device, cfg)
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{cfg.epochs}  train={train_loss:.4f}  val={val_loss:.4f}  lr={current_lr:.2e}")

    with open(log_path, "a") as f:
        f.write(f"{epoch},{train_loss:.6f},{val_loss:.6f},{current_lr:.2e}\n")

    if epoch % cfg.save_every == 0:
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"vae_epoch_{epoch:04d}.pt")
        save_checkpoint(model, optimizer, scheduler, epoch, val_loss, ckpt_path)

final_path = os.path.join(cfg.checkpoint_dir, "vae_final.pt")
save_checkpoint(model, optimizer, scheduler, cfg.epochs, val_loss, final_path)

## Visualize Reconstructions

Run after training (or after loading a checkpoint) to compare originals with their VAE reconstructions.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, device, n=8):
    model.eval()
    batch = next(iter(loader))[:n].to(device)
    recon, _, _ = model(batch)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.6, 3.5))
    for i in range(n):
        orig = batch[i].cpu().permute(1, 2, 0).numpy()
        rec  = recon[i].cpu().permute(1, 2, 0).numpy()
        axes[0, i].imshow(orig[:, :, :3])
        axes[0, i].axis("off")
        axes[1, i].imshow(rec[:, :, :3])
        axes[1, i].axis("off")

    axes[0, 0].set_title("Original",      fontsize=9, loc="left")
    axes[1, 0].set_title("Reconstructed", fontsize=9, loc="left")
    plt.tight_layout()
    plt.show()

show_reconstructions(model, test_loader, device)

## Loss Curve

In [ ]:
epochs_log, train_losses, val_losses, lrs = [], [], [], []
with open(log_path) as f:
    for row in csv.DictReader(f):
        epochs_log.append(int(row["epoch"]))
        train_losses.append(float(row["train_loss"]))
        val_losses.append(float(row["val_loss"]))
        lrs.append(float(row["lr"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_log, train_losses, label="Train")
ax1.plot(epochs_log, val_losses,   label="Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("VAE Training Loss")
ax1.legend()

ax2.plot(epochs_log, lrs, color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Learning Rate")
ax2.set_title("Learning Rate Schedule")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()